<a href="https://colab.research.google.com/github/just-Naki-here/psychxcodenamechart2osumania/blob/main/psych_codename_chart2osu!mania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Imports, Uploads, and Cleanup
# CLEANUP + SETUP

import os
import json
import math

INPUT_FOLDER = "."
OUTPUT_FOLDER = "osu_output"

# Create output folder if needed
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Delete previously converted .osu files inside osu_output only
for f in os.listdir(OUTPUT_FOLDER):
    if f.endswith(".osu"):
        os.remove(os.path.join(OUTPUT_FOLDER, f))

print("Old .osu files inside osu_output deleted.")

# Delete previously converted json files (only ones that are empty or previously modified if needed)
for f in os.listdir(INPUT_FOLDER):
    if f.endswith(".json"):
        # You can remove this condition if you want ALL json deleted
        pass  # keeping original jsons intact unless you want full wipe

print("Cleanup complete.")

Old .osu files inside osu_output deleted.
Cleanup complete.


In [ ]:
# @title file upload
import json
import os
from zipfile import ZipFile
from google.colab import files

uploaded = files.upload()
chart_files = [f for f in uploaded if f.endswith(".json")]

print(f"Loaded {len(chart_files)} chart(s)")

Saving corrosion-nakitake.json to corrosion-nakitake.json
Loaded 1 chart(s)


Run the 2 cells above first or else the rest of the code will break

# Chart Parse

This is how psych chart files look(extremely simplified)

```
{
  "song": {
    "bpm": 120,
    "notes": [
      {
        "sectionNotes": [
          [time, lane, sustain, noteType]
        ],
        "mustHitSection": true
      }
    ]
  }
}

```
How codename chart files look


```
"strumLines": [
  {
    "notes": [
      {
        "id": 2,
        "sLen": 348,
        "time": 22384.6153846154,
        "type": 0
      }
    ]
  }
]
```
Meaning of each field

---



Field	Meaning<br>
<br>
<br>
<br>
<br>
id<br>Lane index (0–3 BF / 4–7 opponent in some charts)<br>

---


time	<br>Note time in milliseconds

---
sLen <br>	Sustain length in milliseconds

---
type<br>	Custom note type (0 = normal, others = gimmicks)






In [ ]:
# @title Chart Load/Normalization
def load_fnf_chart(path):
    with open(path, "r") as f:
        data = json.load(f)

    notes_out = []
    bpm = 120

    # ---------------- PSYCH ENGINE ----------------
    if "song" in data and isinstance(data["song"].get("notes"), list):
        song = data["song"]
        bpm = song.get("bpm", bpm)

        for section in song["notes"]:
            for note in section.get("sectionNotes", []):
                time = note[0]
                lane = note[1]
                sustain = note[2] if len(note) > 2 else 0

                # Merge sides
                lane = lane % 4

                notes_out.append({
                    "time": time,
                    "lane": lane,
                    "sustain": sustain
                })

    # ---------------- CODENAME ENGINE ----------------
    elif "strumLines" in data:
        for strum in data["strumLines"]:
            for note in strum.get("notes", []):
                time = note.get("time", 0)
                lane = note.get("id", 0)
                sustain = note.get("sLen", 0)

                # Merge sides + force normal notes
                lane = lane % 4

                notes_out.append({
                    "time": time,
                    "lane": lane,
                    "sustain": sustain
                })

    else:
        print(f"[WARN] Unknown chart format: {path}")
    print(bpm,notes_out)
    return bpm, notes_out



# All the logic n' shit

In [ ]:
# @title FULL CODENAME/FNF to OSU MANIA CONVERTER

import os
import json

KEY_COUNT = 4
COLUMN_WIDTH = 512 // KEY_COUNT

def ms_from_step(step, bpm):
    crochet = 60000 / bpm
    step_crochet = crochet / 4
    return step * step_crochet

def extract_bpm_events(song_data):
    bpm_events = []

    base_bpm = song_data.get("bpm", 120)
    current_bpm = base_bpm

    bpm_events.append({
        "time": 0,
        "bpm": base_bpm
    })

    total_steps = 0

    for section in song_data.get("notes", []):
        steps = section.get("lengthInSteps", 16)

        if section.get("changeBPM", False):
            new_bpm = section.get("bpm", current_bpm)

            if new_bpm != current_bpm:
                # Convert total_steps directly to ms using PREVIOUS bpm
                crochet = 60000 / current_bpm
                step_crochet = crochet / 4

                section_time = round(total_steps * step_crochet)

                bpm_events.append({
                    "time": section_time,
                    "bpm": new_bpm
                })

                current_bpm = new_bpm

        total_steps += steps

    return bpm_events
def convert_chart(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    song = data["song"]
    title = song.get("song", "Converted Song")
    bpm_events = extract_bpm_events(song)

    hit_objects = []

    base_bpm = song.get("bpm", 120)

    for section in song["notes"]:
        section_bpm = section.get("bpm", base_bpm)

        for note in section.get("sectionNotes", []):
            # Format: [time, lane, sustainLength]
            time = int(note[0])
            lane = int(note[1] % KEY_COUNT)
            sustain = int(note[2])

            x = lane * COLUMN_WIDTH + COLUMN_WIDTH // 2
            y = 192

            if sustain > 0:
                end_time = time + sustain
                hit_objects.append(
                    f"{x},{y},{time},128,0,{end_time}:0:0:0:0:\n"
                )
            else:
                hit_objects.append(
                    f"{x},{y},{time},1,0,0:0:0:0:\n"
                )

    output_name = f"{title}.osu"
    output_path = os.path.join(OUTPUT_FOLDER, output_name)

    with open(output_path, "w", encoding="utf-8") as osu:
        osu.write("osu file format v14\n\n")

        osu.write("[General]\n")
        osu.write("AudioFilename: merged_audio.mp3\n")
        osu.write("Mode: 3\n\n")

        osu.write("[Metadata]\n")
        osu.write(f"Title:{title}\n")
        osu.write("Artist:Converted\n")
        osu.write("Creator:FNF Converter\n")
        osu.write("Version:Hard\n\n")

        osu.write("[Difficulty]\n")
        osu.write(f"CircleSize:{KEY_COUNT}\n")
        osu.write("HPDrainRate:5\n")
        osu.write("OverallDifficulty:8\n")
        osu.write("ApproachRate:5\n")
        osu.write("SliderMultiplier:1.4\n")
        osu.write("SliderTickRate:1\n\n")

        osu.write("[TimingPoints]\n")

        for event in bpm_events:
            time = int(event["time"])
            bpm = float(event["bpm"])
            beat_length = 60000 / bpm
            osu.write(f"{time},{beat_length},4,2,0,100,1,0\n")

        osu.write("\n[HitObjects]\n")

        for obj in sorted(hit_objects, key=lambda x: int(x.split(",")[2])):
            osu.write(obj)

    print(f"Converted: {output_name}")


# BATCH CONVERT
for file in os.listdir(INPUT_FOLDER):
    if file.endswith(".json"):
        convert_chart(os.path.join(INPUT_FOLDER, file))

        # Delete json after conversion
        os.remove(os.path.join(INPUT_FOLDER, file))

print("All charts converted.")



Converted: corrosion-nakitake.osu
All charts converted.


# Optional

If you want to convert the inst and the voices files zip them together for faster upload

In [ ]:
# @title If you want to convert the voices.ogg and inst.ogg files together run this cell
!apt-get install ffmpeg -y
!pip install pydub

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
# @title Fast ZIP Upload (Inst + Voices → MP3)
# @markdown OPTIONAL: Inst + Voices → MP3
# Supports:
# @markdown 1) Sidebar files: drag & drop
# @markdown 2) ZIP upload: to use this part, you *must* zip the ***Inst.ogg*** and the ***Voices.ogg*** files together *then* upload it when prompted<br>it will still take around 4+ minutes depending on the length, quality, and size of the zip file, as well as your internet speed
# @markdown 3) Google Drive manual path input

import os
import zipfile
from pydub import AudioSegment
from google.colab import files, drive
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)

def find_audio_files(search_path="."):
    inst_file = None
    voices_file = None

    for root, _, filenames in os.walk(search_path):
        for f in filenames:
            name_lower = f.lower()
            full_path = os.path.join(root, f)

            if "inst" in name_lower and f.lower().endswith((".ogg", ".wav", ".mp3")):
                inst_file = full_path
            elif "voice" in name_lower and f.lower().endswith((".ogg", ".wav", ".mp3")):
                voices_file = full_path

    return inst_file, voices_file


print("Choose audio input method:")
print("1 = Sidebar files")
print("2 = ZIP upload")
print("3 = Google Drive (manual paths)")
print("Anything else = Skip")

choice = input("Enter choice: ").strip()

inst_file = None
voices_file = None


# OPTION 1: SIDEBAR
if choice == "1":
    inst_file, voices_file = find_audio_files(".")
    if not inst_file or not voices_file:
        print("Inst and Voices not found in sidebar.")

# OPTION 2: ZIP UPLOAD
elif choice == "2":
    print("Upload a ZIP containing Inst and Voices.")
    uploaded = files.upload()

    if uploaded:
        zip_name = list(uploaded.keys())[0]

        if zip_name.lower().endswith(".zip"):
            extract_folder = "audio_temp"

            if os.path.isdir(extract_folder):
                for f in os.listdir(extract_folder):
                    os.remove(os.path.join(extract_folder, f))
            else:
                os.makedirs(extract_folder)

            with zipfile.ZipFile(zip_name, 'r') as zip_ref:
                zip_ref.extractall(extract_folder)

            inst_file, voices_file = find_audio_files(extract_folder)
        else:
            print("Uploaded file is not a ZIP.")

# OPTION 3: GOOGLE DRIVE
elif choice == "3":
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    inst_path = input("Enter full path to Inst file: ").strip()
    voices_path = input("Enter full path to Voices file: ").strip()

    if os.path.exists(inst_path) and os.path.exists(voices_path):
        inst_file = inst_path
        voices_file = voices_path
    else:
        print("One or both file paths are invalid.")

else:
    print("Skipping audio merge.")


# MERGE IF VALID
if inst_file and voices_file:
    print("Merging audio...")

    inst = AudioSegment.from_file(inst_file)
    voices = AudioSegment.from_file(voices_file)

    # Match lengths
    if len(voices) < len(inst):
        voices += AudioSegment.silent(duration=len(inst) - len(voices))
    elif len(inst) < len(voices):
        inst += AudioSegment.silent(duration=len(voices) - len(inst))

    merged = inst.overlay(voices)

    output_name = "merged_audio.mp3"
    merged.export(output_name, format="mp3", bitrate="320k")

    print(f"[OK] Created {output_name}")
    files.download(output_name)

else:
    print("Audio merge not completed.")

Choose audio input method:
1 = Sidebar files
2 = ZIP upload
3 = Google Drive (manual paths)
Anything else = Skip
Enter choice: 3
Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter full path to Inst file: /content/drive/MyDrive/tesstttt/Inst.ogg
Enter full path to Voices file: /content/drive/MyDrive/tesstttt/Voices.ogg
Merging audio...
[OK] Created merged_audio.mp3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>